# LeetCode 1564 — Put Boxes Into the Warehouse I bằng Python

Link đề: https://leetcode.com/problems/put-boxes-into-the-warehouse-i/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

> **Ghi chú Premium:** đây là bài Premium (🔒). Đề bài và chữ ký hàm dưới đây được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1564/). Nếu có tài khoản Premium, nên đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
class Solution:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá luật "đẩy hộp từ trái sang, bị chặn bởi phòng thấp hơn" thành **chiều cao hiệu dụng** `eff[i] = min(warehouse[0..i])` (min tiền tố).
2. Viết baseline **vét cạn mọi thứ tự đẩy và vị trí đặt** cho input nhỏ để có chuẩn đối chiếu đáng tin.
3. Nắm hai lời giải tham lam `O(n log n)`: (a) hộp **nhỏ nhất** vào phòng **sâu nhất** còn vừa (hai con trỏ, theo doocs); (b) hộp **lớn nhất** vào phòng **ngoài cùng** — không cần tiền xử lý.
4. Chứng minh tham lam bằng lập luận **hoán đổi** và cận trên.
5. Tránh các bẫy: so hộp với `warehouse[i]` thay vì min tiền tố, ghép hộp nhỏ vào phòng nông, quên dừng khi hết phòng.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên dương `boxes` và `warehouse`: chiều cao của các hộp (bề rộng 1 đơn vị) và chiều cao của `n` phòng trong một nhà kho. Các phòng đánh số `0..n−1` từ trái sang phải, phòng `i` cao `warehouse[i]`.

Luật đưa hộp vào kho:
- Không được **chồng** hộp lên nhau (mỗi phòng tối đa 1 hộp).
- Được **sắp lại thứ tự** đẩy hộp.
- Hộp chỉ được đẩy vào **từ trái sang phải**.
- Nếu một phòng thấp hơn chiều cao hộp thì hộp đó (và mọi hộp phía sau nó) bị chặn **trước** phòng đó.

Trả về **số hộp tối đa** có thể đưa vào kho.

**Ví dụ 1:** `boxes = [4,3,4,1]`, `warehouse = [5,3,3,4,1]` → `3`. Đặt hộp cao 1 vào phòng 4; hộp cao 3 vào một trong các phòng 1, 2, 3; một hộp cao 4 vào phòng 0. Không thể đưa cả 4 hộp.

**Ví dụ 2:** `boxes = [1,2,2,3,4]`, `warehouse = [3,4,1,2]` → `3`. Hộp cao 4 không qua được phòng 0 (cao 3). Hai phòng cuối (cao 1 và 2, nhưng bị chặn bởi phòng cao 1) chỉ chứa được hộp cao 1.

**Ví dụ 3:** `boxes = [1,2,3]`, `warehouse = [1,2,3,4]` → `1`. Phòng đầu cao 1 nên chỉ hộp cao 1 vào được.

**Ràng buộc:** `n == warehouse.length`, `1 ≤ boxes.length, warehouse.length ≤ 10⁵`, `1 ≤ boxes[i], warehouse[i] ≤ 10⁹`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import accumulate
from typing import List

examples = [
    ([4, 3, 4, 1], [5, 3, 3, 4, 1], 3),
    ([1, 2, 2, 3, 4], [3, 4, 1, 2], 3),
    ([1, 2, 3], [1, 2, 3, 4], 1),
]
for boxes, wh, want in examples:
    print(f"boxes = {boxes}, warehouse = {wh} -> mong đợi {want}")

boxes = [4, 3, 4, 1], warehouse = [5, 3, 3, 4, 1] -> mong đợi 3
boxes = [1, 2, 2, 3, 4], warehouse = [3, 4, 1, 2] -> mong đợi 3
boxes = [1, 2, 3], warehouse = [1, 2, 3, 4] -> mong đợi 1


## 3. Kiến thức nền

### 3.1 Min tiền tố với `itertools.accumulate`

`accumulate(a, min)` trả về `[a[0], min(a[0],a[1]), min(a[0],a[1],a[2]), ...]`. Đây chính là "chiều cao lớn nhất mà một hộp có thể có để tới được phòng `i`".

In [2]:
wh = [5, 3, 3, 4, 1]
eff = list(accumulate(wh, min))
print("warehouse:", wh)
print("eff      :", eff)

warehouse: [5, 3, 3, 4, 1]
eff      : [5, 3, 3, 3, 1]


### 3.2 Sắp xếp tăng / giảm

`sorted(a)` tăng dần, `sorted(a, reverse=True)` giảm dần. `list.sort()` sắp tại chỗ (thay đổi input — LeetCode chấp nhận, nhưng khi test nhiều cách trên cùng dữ liệu nên truyền bản sao).

In [3]:
boxes = [4, 3, 4, 1]
print(sorted(boxes), sorted(boxes, reverse=True), boxes)

[1, 3, 4, 4] [4, 4, 3, 1] [4, 3, 4, 1]


### 3.3 Hai con trỏ chạy ngược chiều

Con trỏ `i` chạy trên hộp (từ nhỏ tới lớn), con trỏ `j` chạy trên phòng (từ sâu ra ngoài). Mỗi con trỏ chỉ đi một chiều ⇒ tổng `O(len(boxes) + n)` sau khi sắp.

In [4]:
a, b = [1, 3, 4, 4], [5, 3, 3, 3, 1]       # hộp tăng dần, eff (không tăng)
i, j = 0, len(b) - 1
pairs = []
while i < len(a) and j >= 0:
    if b[j] >= a[i]:
        pairs.append((a[i], j)); i += 1
    j -= 1
print(pairs)

[(1, 4), (3, 3), (4, 0)]


### 3.4 `lru_cache` cho vét cạn có nhớ

Hàm đệ quy với tham số băm được (số nguyên, tuple) có thể được nhớ kết quả bằng `@lru_cache(None)`, tránh tính lại cùng một trạng thái.

In [5]:
@lru_cache(None)
def fib(k):
    return k if k < 2 else fib(k - 1) + fib(k - 2)
print(fib(80))

23416728348467685


### 3.5 Bitmask để đánh dấu tập hộp đã dùng

`mask >> b & 1` kiểm tra hộp `b` đã dùng; `mask | (1 << b)` đánh dấu dùng. Dùng trong vét cạn.

In [6]:
mask = 0
mask |= 1 << 2
print(bin(mask), [(b, mask >> b & 1) for b in range(4)])

0b100 [(0, 0), (1, 0), (2, 1), (3, 0)]


## 4. Ý tưởng nền tảng / trực giác

**Quan sát 1 — phòng sâu bị "bóp" bởi phòng nông.** Muốn tới phòng `i`, hộp phải chui qua các phòng `0..i`. Vậy phòng `i` thực chất chỉ chứa được hộp cao ≤ `eff[i] = min(warehouse[0..i])`. Sau khi thay `warehouse` bằng `eff`, mảng **không tăng** từ trái sang phải.

Ví dụ 1: `warehouse = [5,3,3,4,1]` ⇒ `eff = [5,3,3,3,1]` (phòng 3 cao 4 nhưng bị phòng 1–2 cao 3 chặn).

**Quan sát 2 — đẩy hộp vào phòng sâu trước.** Hộp đặt vào phòng `j` sẽ chặn mọi hộp sau muốn đi sâu hơn `j`. Nên ta đẩy hộp dành cho phòng **sâu nhất** trước, rồi tới phòng nông hơn. Khi đó chỉ còn ràng buộc `hộp ≤ eff[phòng]`.

**Tính tay ví dụ 1:** hộp tăng dần `[1, 3, 4, 4]`, `eff = [5, 3, 3, 3, 1]`.
- Hộp 1: phòng sâu nhất là 4 (`eff = 1 ≥ 1`) ✔ → phòng 4.
- Hộp 3: phòng 3 (`eff = 3`) ✔ → phòng 3.
- Hộp 4: phòng 2 (`3 < 4`) ✘, phòng 1 ✘, phòng 0 (`5 ≥ 4`) ✔ → phòng 0.
- Hộp 4 còn lại: hết phòng phía ngoài ⇒ dừng. Đáp án `3`.

In [7]:
for boxes, wh, want in examples:
    print(f"warehouse={wh} -> eff={list(accumulate(wh, min))}, hộp tăng dần={sorted(boxes)}, mong đợi {want}")

warehouse=[5, 3, 3, 4, 1] -> eff=[5, 3, 3, 3, 1], hộp tăng dần=[1, 3, 4, 4], mong đợi 3
warehouse=[3, 4, 1, 2] -> eff=[3, 3, 1, 1], hộp tăng dần=[1, 2, 2, 3, 4], mong đợi 3
warehouse=[1, 2, 3, 4] -> eff=[1, 1, 1, 1], hộp tăng dần=[1, 2, 3], mong đợi 1


## 5. Cách 1 — Brute Force / baseline: vét cạn thứ tự và vị trí

**Ý tưởng.** Mô phỏng đúng luật: trạng thái = (tập hộp đã dùng, các phòng đã có hộp). Ở mỗi bước, chọn một hộp chưa dùng, đẩy từ trái: nó có thể dừng ở **bất kỳ** phòng nào trên đường đi, chừng nào phòng chưa bị chiếm và đủ cao (không vượt qua được phòng đã có hộp hoặc phòng thấp hơn nó). Lấy max số hộp. Có nhớ bằng `lru_cache`.

**Độ phức tạp:** số trạng thái tới `2^{len(boxes)} · 2^n` — chỉ dùng với `len(boxes), n ≤ 6`. Đây là chuẩn đối chiếu **trực tiếp theo đề**, không dựa vào bất kỳ nhận xét nào ở mục 4.

In [8]:
def brute_force(boxes, warehouse, both_sides=False):
    n, k = len(warehouse), len(boxes)

    @lru_cache(None)
    def go(used, occ):
        best = 0
        for b in range(k):
            if used >> b & 1:
                continue
            h = boxes[b]
            reach = set()
            i = 0
            while i < n and not occ[i] and warehouse[i] >= h:      # đẩy từ trái
                reach.add(i)
                i += 1
            if both_sides:                                         # (dùng cho bài tập 5: đẩy từ phải)
                i = n - 1
                while i >= 0 and not occ[i] and warehouse[i] >= h:
                    reach.add(i)
                    i -= 1
            for p in reach:
                best = max(best, 1 + go(used | (1 << b), occ[:p] + (True,) + occ[p + 1:]))
        return best

    return go(0, (False,) * n)

for boxes, wh, want in examples:
    print(brute_force(boxes, wh), "mong đợi", want)

3 mong đợi 3
3 mong đợi 3
1 mong đợi 1


In [9]:
# Nút thắt: số trạng thái bùng nổ
rng = random.Random(0)
for k, n in [(5, 5), (6, 6), (7, 7), (8, 8), (9, 9)]:
    boxes = [rng.randint(1, 3) for _ in range(k)]       # hộp thấp, kho cao: hộp nào cũng vừa -> nhiều trạng thái nhất
    wh = [rng.randint(4, 8) for _ in range(n)]
    t0 = time.perf_counter()
    r = brute_force(boxes, wh)
    print(f"{k} hộp, {n} phòng: {r} ({time.perf_counter() - t0:.3f}s)")

5 hộp, 5 phòng: 5 (0.001s)
6 hộp, 6 phòng: 6 (0.003s)
7 hộp, 7 phòng: 7 (0.010s)
8 hộp, 8 phòng: 8 (0.051s)


9 hộp, 9 phòng: 9 (0.202s)


**Nút thắt:** ta thử mọi thứ tự và mọi vị trí dừng. Nhưng theo mục 4, chỉ cần quyết định **hộp nào ghép với phòng nào** (với ràng buộc `hộp ≤ eff`) — thứ tự đẩy luôn có thể chọn "phòng sâu trước".

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — quy về ghép cặp.** Một tập ghép (hộp `b` → phòng `p`, các phòng khác nhau) là **thực hiện được** ⇔ mọi cặp thoả `b ≤ eff[p]`. (⇒) hộp tới được `p` thì qua được các phòng `0..p`. (⇐) đẩy theo thứ tự phòng giảm dần: hộp vào phòng sâu nhất trước, đường đi lúc đó trống.

**Insight 2 — tham lam "nhỏ vào sâu".** Sắp hộp **tăng dần**; con trỏ phòng `j` bắt đầu từ phòng sâu nhất. Với hộp nhỏ nhất còn lại: lùi `j` cho tới phòng đầu tiên có `eff[j] ≥ hộp`, đặt hộp ở đó, `j −= 1`. *Nếu* `j < 0` *thì* dừng. Lý do: hộp nhỏ nhất vừa với nhiều phòng nhất — cho nó phòng sâu nhất (phòng "kém" nhất vì `eff` không tăng) để dành phòng cao cho hộp lớn.

**Insight 3 — tham lam "lớn vào ngoài".** Sắp hộp **giảm dần**; con trỏ phòng `i` từ 0. Với mỗi hộp: *nếu* `hộp ≤ warehouse[i]` *thì* đặt vào phòng `i`, `i += 1`; ngược lại **bỏ hộp đó** (nó quá to cho phòng ngoài cùng còn trống, mà mọi phòng sâu hơn còn bị chặn bởi phòng này). Không cần `eff`: các phòng `0..i−1` đã chứa hộp lớn hơn hoặc bằng hộp hiện tại, nên chúng đủ cao cho hộp hiện tại đi qua.

**Invariant (Insight 2):** sau khi xử lý `t` hộp nhỏ nhất, chúng chiếm `t` phòng sâu nhất có thể theo thứ tự, và mọi phòng nông hơn `j` vẫn trống.

In [10]:
wh = [5, 3, 3, 4, 1]
eff = list(accumulate(wh, min))
for p in range(len(wh)):
    print(f"phòng {p}: cao {wh[p]}, hiệu dụng {eff[p]}" + ("  <- bị chặn bởi phòng nông hơn" if eff[p] < wh[p] else ""))

phòng 0: cao 5, hiệu dụng 5
phòng 1: cao 3, hiệu dụng 3
phòng 2: cao 3, hiệu dụng 3
phòng 3: cao 4, hiệu dụng 3  <- bị chặn bởi phòng nông hơn
phòng 4: cao 1, hiệu dụng 1


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Min tiền tố + hộp nhỏ nhất vào phòng sâu nhất (lời giải doocs)

Từng đoạn:
1. `left[i] = min(left[i−1], warehouse[i])` — chiều cao hiệu dụng.
2. `boxes.sort()` — tăng dần.
3. `i` duyệt hộp, `j` từ `n − 1`: lùi `j` khi `left[j] < boxes[i]`; nếu `j < 0` thì dừng; ngược lại đặt hộp (`i += 1`, `j −= 1`).
4. Trả `i` (số hộp đã đặt).

In [11]:
def small_to_deep(boxes, warehouse):
    n = len(warehouse)
    left = list(accumulate(warehouse, min))
    bs = sorted(boxes)
    i, j = 0, n - 1
    while i < len(bs):
        while j >= 0 and left[j] < bs[i]:
            j -= 1
        if j < 0:
            break
        i, j = i + 1, j - 1
    return i

for boxes, wh, want in examples:
    print(small_to_deep(boxes, wh), "mong đợi", want)

3 mong đợi 3
3 mong đợi 3
1 mong đợi 1


Code nộp LeetCode cho cách 2 (đúng chữ ký doocs):

In [12]:
class SolutionDoocs:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        n = len(warehouse)
        left = [warehouse[0]] * n
        for i in range(1, n):
            left[i] = min(left[i - 1], warehouse[i])
        boxes.sort()
        i, j = 0, n - 1
        while i < len(boxes):
            while j >= 0 and left[j] < boxes[i]:
                j -= 1
            if j < 0:
                break
            i, j = i + 1, j - 1
        return i

print([SolutionDoocs().maxBoxesInWarehouse(list(b), list(w)) for b, w, _ in examples])

[3, 3, 1]


**Độ phức tạp:** Time `O(n + k log k)` với `k = len(boxes)`, Space `O(n)` cho `left`.

### 7.2 Cách 3 — Hộp lớn nhất vào phòng ngoài cùng (không cần min tiền tố)

Từng đoạn:
1. Sắp hộp giảm dần; `i = 0` là phòng ngoài cùng còn trống.
2. Với mỗi hộp `h`: nếu `i < n` và `h ≤ warehouse[i]` thì đặt (`i += 1`), ngược lại bỏ qua hộp.
3. Trả `i`.

Tại sao không cần `eff`? Hộp đặt vào phòng `i` phải qua phòng `0..i−1`; các phòng đó chứa hộp đã đặt trước, cao **≥** `h` (sắp giảm dần) và mỗi phòng cao ≥ hộp của nó ⇒ đủ cao cho `h`. (Thứ tự đẩy thực tế: đẩy hộp của phòng sâu nhất trước.)

In [13]:
class SolutionDesc:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        n = len(warehouse)
        i = 0
        for h in sorted(boxes, reverse=True):
            if i < n and h <= warehouse[i]:
                i += 1
        return i

print([SolutionDesc().maxBoxesInWarehouse(list(b), list(w)) for b, w, _ in examples])

[3, 3, 1]


**Độ phức tạp:** Time `O(k log k + k)`, Space `O(k)` (bản sao đã sắp) hoặc `O(1)` nếu sắp tại chỗ.

### 7.3 Cách 4 — Ghép hai mảng đã sắp (kiểu "Assign Cookies")

`eff` không tăng, nên `eff[::-1]` là dãy **không giảm**. Bài toán thành: ghép tối đa hộp với "chỗ chứa" sao cho `hộp ≤ chỗ chứa` — tham lam hai con trỏ kinh điển: duyệt chỗ chứa tăng dần, mỗi chỗ nhận hộp nhỏ nhất còn lại nếu vừa.

In [14]:
class SolutionMatch:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        caps = list(accumulate(warehouse, min))[::-1]     # tăng dần
        bs = sorted(boxes)
        b = 0
        for cap in caps:
            if b < len(bs) and bs[b] <= cap:
                b += 1
        return b

print([SolutionMatch().maxBoxesInWarehouse(list(b), list(w)) for b, w, _ in examples])

[3, 3, 1]


**Độ phức tạp:** `O(n + k log k)`, Space `O(n + k)`. Đây chính là cách 2 viết lại theo góc nhìn "chỗ chứa".

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2: `boxes = [1,2,2,3,4]`, `warehouse = [3,4,1,2]` ⇒ `left = [3,3,1,1]`.

| Hộp `bs[i]` | j trước | Lùi j vì `left[j] < hộp` | j đặt | Kết quả |
|-------------|---------|---------------------------|-------|---------|
| 1 | 3 | — | 3 | hộp 1 → phòng 3 |
| 2 | 2 | `left[2]=1<2` ⇒ j=1 | 1 | hộp 2 → phòng 1 |
| 2 | 0 | — | 0 | hộp 2 → phòng 0 |
| 3 | −1 | | | hết phòng ⇒ dừng, đáp án **3** |

In [15]:
def trace_small_to_deep(boxes, warehouse):
    n = len(warehouse)
    left = list(accumulate(warehouse, min))
    bs = sorted(boxes)
    print(f"left = {left}, hộp tăng dần = {bs}")
    i, j = 0, n - 1
    while i < len(bs):
        skipped = []
        while j >= 0 and left[j] < bs[i]:
            skipped.append(j)
            j -= 1
        if j < 0:
            print(f"  hộp {bs[i]}: bỏ qua phòng {skipped} -> hết phòng, dừng")
            break
        print(f"  hộp {bs[i]}: bỏ qua phòng {skipped}, đặt vào phòng {j} (left={left[j]})")
        i, j = i + 1, j - 1
    print("  đáp án", i)
    return i

trace_small_to_deep([1, 2, 2, 3, 4], [3, 4, 1, 2])

left = [3, 3, 1, 1], hộp tăng dần = [1, 2, 2, 3, 4]
  hộp 1: bỏ qua phòng [], đặt vào phòng 3 (left=1)
  hộp 2: bỏ qua phòng [2], đặt vào phòng 1 (left=3)
  hộp 2: bỏ qua phòng [], đặt vào phòng 0 (left=3)
  hộp 3: bỏ qua phòng [] -> hết phòng, dừng
  đáp án 3


3

In [16]:
trace_small_to_deep([4, 3, 4, 1], [5, 3, 3, 4, 1])

left = [5, 3, 3, 3, 1], hộp tăng dần = [1, 3, 4, 4]
  hộp 1: bỏ qua phòng [], đặt vào phòng 4 (left=1)
  hộp 3: bỏ qua phòng [], đặt vào phòng 3 (left=3)
  hộp 4: bỏ qua phòng [2, 1], đặt vào phòng 0 (left=5)
  hộp 4: bỏ qua phòng [] -> hết phòng, dừng
  đáp án 3


3

Trace cách 3 (lớn vào ngoài) trên ví dụ 1:

In [17]:
def trace_desc(boxes, warehouse):
    n = len(warehouse)
    i = 0
    for h in sorted(boxes, reverse=True):
        if i < n and h <= warehouse[i]:
            print(f"  hộp {h}: phòng {i} cao {warehouse[i]} -> đặt")
            i += 1
        else:
            reason = "hết phòng" if i >= n else f"phòng {i} chỉ cao {warehouse[i]}"
            print(f"  hộp {h}: {reason} -> bỏ hộp")
    print("  đáp án", i)
    return i

trace_desc([4, 3, 4, 1], [5, 3, 3, 4, 1])

  hộp 4: phòng 0 cao 5 -> đặt
  hộp 4: phòng 1 chỉ cao 3 -> bỏ hộp
  hộp 3: phòng 1 cao 3 -> đặt
  hộp 1: phòng 2 cao 3 -> đặt
  đáp án 3


3

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def maxBoxesInWarehouse(self, boxes: List[int], warehouse: List[int]) -> int:
        n = len(warehouse)
        left = [warehouse[0]] * n                 # left[i] = min(warehouse[0..i])
        for i in range(1, n):
            left[i] = min(left[i - 1], warehouse[i])
        boxes.sort()
        i, j = 0, n - 1                           # hộp nhỏ nhất -> phòng sâu nhất còn vừa
        while i < len(boxes):
            while j >= 0 and left[j] < boxes[i]:
                j -= 1
            if j < 0:
                break
            i, j = i + 1, j - 1
        return i

for boxes, wh, want in examples:
    got = Solution().maxBoxesInWarehouse(list(boxes), list(wh))
    print(got, want)
    assert got == want

3 3
3 3
1 1


## 10. Vì sao thuật toán đúng

**(a) Quy về ghép cặp** (Insight 1): một tập hộp đặt được ⇔ có cách ghép mỗi hộp với một phòng riêng mà `hộp ≤ eff[phòng]`.

**(b) Cận trên & tối ưu của "nhỏ vào sâu".** Giả sử một phương án tối ưu đặt `t` hộp. Thay các hộp đó bằng `t` hộp **nhỏ nhất** (vẫn vừa các phòng cũ) ⇒ có phương án tối ưu dùng `t` hộp nhỏ nhất. Trong phương án đó, sắp hộp tăng dần và phòng theo độ sâu giảm dần rồi ghép theo thứ tự (hộp nhỏ hơn ↔ phòng sâu hơn): vì `eff` không tăng, phép "gỡ chéo" (nếu hộp nhỏ ở phòng nông còn hộp lớn ở phòng sâu thì hoán đổi) giữ tính hợp lệ. Tham lam của ta với mỗi hộp chọn phòng **sâu nhất có thể**, nên vị trí phòng của hộp thứ `r` (theo tham lam) luôn **sâu hơn hoặc bằng** vị trí trong phương án tối ưu đã gỡ chéo (quy nạp theo `r`) ⇒ tham lam không bao giờ hết phòng sớm hơn ⇒ đặt được ít nhất `t` hộp.

**(c) "Lớn vào ngoài".** Hộp bị bỏ ở cách 3 là hộp `h > warehouse[i]` với `i` là phòng ngoài cùng còn trống: mọi phòng sâu hơn `i` có `eff ≤ warehouse[i] < h`, các phòng `< i` đã có hộp ⇒ không phương án nào (giữ nguyên các hộp lớn hơn đã đặt) đặt thêm được `h`; lập luận hoán đổi chuẩn cho thấy giữ các hộp lớn đã đặt không làm mất tối ưu. Kiểm chứng thực nghiệm với vét cạn ở mục 12.

In [19]:
# thực nghiệm (a): "đặt được" theo luật đẩy == "ghép được" theo eff, trên các bộ nhỏ
rng = random.Random(10)
for _ in range(300):
    boxes = [rng.randint(1, 6) for _ in range(rng.randint(1, 5))]
    wh = [rng.randint(1, 7) for _ in range(rng.randint(1, 5))]
    eff = list(accumulate(wh, min))
    # ghép tối đa theo eff bằng vét cạn: mọi tập hộp S, mọi cách gán S vào các phòng khác nhau
    from itertools import permutations
    best = 0
    k = len(boxes)
    for mask in range(1 << k):
        S = [boxes[t] for t in range(k) if mask >> t & 1]
        if len(S) <= len(wh) and len(S) > best:
            if any(all(h <= eff[p] for h, p in zip(S, rooms)) for rooms in permutations(range(len(wh)), len(S))):
                best = len(S)
    assert best == brute_force(boxes, wh)
print("luật đẩy hộp == ghép cặp theo eff: OK")

luật đẩy hộp == ghép cặp theo eff: OK


## 11. Phân tích Time / Space

Gọi `n = len(warehouse)`, `k = len(boxes)`.

| Cách | Time | Space |
|------|------|-------|
| Vét cạn | `O(2^k · 2^n · k · n)` | như trên |
| Min tiền tố + nhỏ vào sâu | `O(n + k log k)` | `O(n)` |
| Lớn vào ngoài | `O(k log k)` | `O(1)`–`O(k)` |
| Ghép hai mảng | `O(n + k log k)` | `O(n + k)` |

Phần trội là sắp xếp hộp. Cách 3 thậm chí không đọc hết `warehouse` nếu hộp hết trước.

In [20]:
rng = random.Random(1564)
boxes_big = [rng.randint(1, 10**9) for _ in range(10**5)]
wh_big = [rng.randint(1, 10**9) for _ in range(10**5)]
wh_slow = sorted(wh_big, reverse=True)                    # kho "dốc xuống": không bị chặn đột ngột
for name, S in [("doocs", SolutionDoocs), ("desc", SolutionDesc), ("match", SolutionMatch), ("Solution", Solution)]:
    t0 = time.perf_counter()
    r1 = S().maxBoxesInWarehouse(list(boxes_big), list(wh_big))
    r2 = S().maxBoxesInWarehouse(list(boxes_big), list(wh_slow))
    print(f"{name:8s}: {r1}, {r2} ({time.perf_counter() - t0:.3f}s)")

doocs   : 944, 99848 (0.084s)


desc    : 944, 99848 (0.067s)
match   : 944, 99848 (0.085s)


Solution: 944, 99848 (0.078s)


## 12. Bộ test

Lưu ý: `SolutionDoocs` và `Solution` sắp `boxes` tại chỗ, nên luôn truyền **bản sao** (`list(...)`).

In [21]:
METHODS = {
    "brute_force": brute_force,
    "small_to_deep": small_to_deep,
    "SolutionDoocs": lambda b, w: SolutionDoocs().maxBoxesInWarehouse(list(b), list(w)),
    "SolutionDesc": lambda b, w: SolutionDesc().maxBoxesInWarehouse(list(b), list(w)),
    "SolutionMatch": lambda b, w: SolutionMatch().maxBoxesInWarehouse(list(b), list(w)),
    "Solution": lambda b, w: Solution().maxBoxesInWarehouse(list(b), list(w)),
}
for boxes, wh, want in examples:
    for name, f in METHODS.items():
        assert f(boxes, wh) == want, name
print("ví dụ đề: OK")

ví dụ đề: OK


In [22]:
# edge case
edge = [
    ([1], [1], 1),                         # vừa khít
    ([2], [1], 0),                         # không vào được
    ([4], [3, 5], 0),                      # phòng sau cao nhưng bị phòng đầu chặn
    ([1, 1, 1], [1, 1], 2),                # thừa hộp
    ([1], [5, 5, 5], 1),                   # thừa phòng
    ([3, 3, 3], [3, 3, 3], 3),
    ([5, 1, 1], [5, 1, 1], 3),
    ([2, 2], [1, 5, 5], 0),
    ([10**9] * 3, [10**9] * 2, 2),
]
for boxes, wh, want in edge:
    for name, f in METHODS.items():
        assert f(boxes, wh) == want, (name, boxes, wh, f(boxes, wh))
print("edge case: OK")

edge case: OK


In [23]:
# ngẫu nhiên có seed: đối chiếu vét cạn theo luật
rng = random.Random(2017)
for _ in range(1200):
    k, n = rng.randint(1, 5), rng.randint(1, 5)
    lim = rng.choice([3, 6, 10])
    boxes = [rng.randint(1, lim) for _ in range(k)]
    wh = [rng.randint(1, lim) for _ in range(n)]
    ref = brute_force(boxes, wh)
    for name, f in METHODS.items():
        assert f(boxes, wh) == ref, (name, boxes, wh)
print("1200 test ngẫu nhiên (seed 2017): mọi cách khớp vét cạn")

1200 test ngẫu nhiên (seed 2017): mọi cách khớp vét cạn


In [24]:
# cỡ lớn hơn: các cách nhanh khớp nhau
rng = random.Random(6)
for _ in range(300):
    boxes = [rng.randint(1, 50) for _ in range(rng.randint(1, 200))]
    wh = [rng.randint(1, 50) for _ in range(rng.randint(1, 200))]
    ref = small_to_deep(boxes, wh)
    for name in ["SolutionDoocs", "SolutionDesc", "SolutionMatch", "Solution"]:
        assert METHODS[name](boxes, wh) == ref
print("300 test cỡ vừa: OK")

300 test cỡ vừa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So hộp với `warehouse[j]` thay vì min tiền tố

Quên rằng hộp phải **đi qua** các phòng phía trước.

In [25]:
def wrong_no_prefix_min(boxes, warehouse):
    n = len(warehouse)
    bs = sorted(boxes)
    i, j = 0, n - 1
    while i < len(bs):
        while j >= 0 and warehouse[j] < bs[i]:      # SAI: phải dùng min tiền tố
            j -= 1
        if j < 0:
            break
        i, j = i + 1, j - 1
    return i

for boxes, wh, want in examples + [([4], [3, 5], 0)]:
    print(boxes, wh, "| sai:", wrong_no_prefix_min(boxes, wh), "| đúng:", want)

[4, 3, 4, 1] [5, 3, 3, 4, 1] | sai: 3 | đúng: 3
[1, 2, 2, 3, 4] [3, 4, 1, 2] | sai: 3 | đúng: 3
[1, 2, 3] [1, 2, 3, 4] | sai: 2 | đúng: 1
[4] [3, 5] | sai: 1 | đúng: 0


Sửa: dùng `left[j] = min(warehouse[0..j])`.

### Lỗi 2 — Hộp nhỏ nhất vào phòng **ngoài cùng** còn vừa

Ghép hộp nhỏ vào phòng nông (phòng "tốt") làm lãng phí; hộp lớn sau đó không còn chỗ.

In [26]:
def wrong_small_to_front(boxes, warehouse):
    eff = list(accumulate(warehouse, min))
    used = [False] * len(eff)
    cnt = 0
    for h in sorted(boxes):
        for p in range(len(eff)):                  # SAI: tìm phòng nông nhất còn vừa
            if not used[p] and eff[p] >= h:
                used[p] = True
                cnt += 1
                break
    return cnt

for boxes, wh in [([1, 2], [3, 1]), ([1, 3], [3, 3, 1]), examples[0][:2]]:
    print(boxes, wh, "| sai:", wrong_small_to_front(boxes, wh), "| đúng:", Solution().maxBoxesInWarehouse(list(boxes), list(wh)))

[1, 2] [3, 1] | sai: 1 | đúng: 2
[1, 3] [3, 3, 1] | sai: 2 | đúng: 2
[4, 3, 4, 1] [5, 3, 3, 4, 1] | sai: 2 | đúng: 3


Sửa: hộp nhỏ nhất vào phòng **sâu nhất** còn vừa (cách 2), hoặc hộp lớn nhất vào phòng ngoài cùng (cách 3).

### Lỗi 3 — Quên `break` khi `j < 0`

Không dừng khi hết phòng ⇒ `i` vẫn tăng ⇒ đếm cả những hộp không có chỗ. (Python cho phép `left[-1]` nên không báo lỗi — còn tệ hơn!)

In [27]:
def wrong_no_break(boxes, warehouse):
    n = len(warehouse)
    left = list(accumulate(warehouse, min))
    bs = sorted(boxes)
    i, j = 0, n - 1
    while i < len(bs):
        while j >= 0 and left[j] < bs[i]:
            j -= 1
        i, j = i + 1, j - 1                         # SAI: thiếu "if j < 0: break"
    return i

for boxes, wh, want in examples + [([1, 1, 1], [1, 1], 2)]:
    print(boxes, wh, "| sai:", wrong_no_break(boxes, wh), "| đúng:", want)

[4, 3, 4, 1] [5, 3, 3, 4, 1] | sai: 4 | đúng: 3
[1, 2, 2, 3, 4] [3, 4, 1, 2] | sai: 5 | đúng: 3
[1, 2, 3] [1, 2, 3, 4] | sai: 3 | đúng: 1
[1, 1, 1] [1, 1] | sai: 3 | đúng: 2


Sửa: thêm `if j < 0: break` trước khi đặt hộp.

### Lỗi 4 — Cách 3 nhưng bỏ **phòng** thay vì bỏ **hộp**

Khi hộp lớn không vừa phòng `i`, người mới hay "nhảy" sang phòng sau — sai, vì hộp không thể vượt qua phòng `i` thấp hơn nó.

In [28]:
def wrong_skip_room(boxes, warehouse):
    n = len(warehouse)
    i = 0
    cnt = 0
    for h in sorted(boxes, reverse=True):
        while i < n and warehouse[i] < h:          # SAI: hộp không thể vượt phòng thấp
            i += 1
        if i < n:
            cnt += 1
            i += 1
    return cnt

for boxes, wh in [([5, 4], [3, 6]), ([4], [3, 5]), examples[2][:2]]:
    print(boxes, wh, "| sai:", wrong_skip_room(boxes, wh), "| đúng:", Solution().maxBoxesInWarehouse(list(boxes), list(wh)))

[5, 4] [3, 6] | sai: 1 | đúng: 0
[4] [3, 5] | sai: 1 | đúng: 0
[1, 2, 3] [1, 2, 3, 4] | sai: 2 | đúng: 1


Sửa: không vừa phòng `i` thì **bỏ hộp**, giữ nguyên `i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đi vào từ một đầu, bị chặn bởi chỗ hẹp" ⇒ **min tiền tố** (từ hai đầu ⇒ max của min tiền tố và min hậu tố).
- Sau khi có "sức chứa hiệu dụng" ⇒ bài toán **ghép cặp tối đa `a ≤ b`** ⇒ sắp cả hai + hai con trỏ (Assign Cookies, LC 455; Boats to Save People, LC 881 cũng cùng họ).
- Thứ tự thao tác được tự do ⇒ thường có một thứ tự "chuẩn" (phòng sâu trước) biến bài toán động thành bài toán tĩnh.
- Khi chọn tham lam: "phần tử khó xếp nhất lấy chỗ tốt nhất" hoặc "phần tử dễ xếp nhất lấy chỗ tệ nhất" — đều đúng nếu chứng minh được.

## 15. Biến thể của bài

1. **Put Boxes Into the Warehouse II (LC 1580)**: đẩy được từ **cả hai phía** ⇒ `eff[i] = max(min tiền tố, min hậu tố)`, rồi ghép như cũ (bài tập 5).
2. **Assign Cookies (LC 455)**: ghép trẻ (độ tham) với bánh (kích thước) — cùng khuôn ghép hai mảng đã sắp.
3. **Boats to Save People (LC 881)**: ghép cặp với ràng buộc tổng.
4. **Hộp có bề rộng khác nhau / được chồng hộp**: trở thành bài khác (knapsack/bin packing), không còn tham lam đơn giản.
5. **Đếm số cách đặt** đạt tối đa — tổ hợp, khó hơn nhiều.

In [29]:
# Biến thể 2 (LC 455) cùng khuôn ghép hai mảng đã sắp
def find_content_children(g, s):
    g, s = sorted(g), sorted(s)
    i = 0
    for size in s:
        if i < len(g) and g[i] <= size:
            i += 1
    return i

print(find_content_children([1, 2, 3], [1, 1]), find_content_children([1, 2], [1, 2, 3]), "(LC 455: 1 và 2)")

1 2 (LC 455: 1 và 2)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn | thử mọi thứ tự + vị trí dừng | mũ | mũ | kiểm chứng, ≤ 6 hộp/phòng |
| 2. Min tiền tố + nhỏ vào sâu | ghép hộp nhỏ nhất với phòng sâu nhất vừa | `O(n + k log k)` | `O(n)` | **bản nộp** (doocs) |
| 3. Lớn vào ngoài | hộp lớn nhất vào phòng ngoài cùng, không vừa thì bỏ | `O(k log k)` | `O(1)` | ngắn nhất, không cần tiền xử lý |
| 4. Ghép hai mảng | `eff` đảo ngược + Assign Cookies | `O(n + k log k)` | `O(n + k)` | dễ mở rộng sang LC 1580 |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxBoxesInWarehouse([1,1,1], [1,1])` và `maxBoxesInWarehouse([5], [4,10])`?

**Bài 2 (tính tay):** Với `boxes = [2,5,3,1]`, `warehouse = [6,4,4,2,3]`, tính `eff`, chạy tay cách 2 và cho đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_prefix_min(warehouse)`.

**Bài 4 (tự viết lại):** Viết `ex4_assignment(boxes, warehouse)` trả về danh sách cặp `(chiều cao hộp, chỉ số phòng)` của một phương án tối ưu. Kiểm tra: mỗi phòng tối đa một hộp, `hộp ≤ eff[phòng]`, số cặp = đáp án.

**Bài 5 (mini challenge):** Viết `ex5_both_sides(boxes, warehouse)` cho phiên bản **đẩy được từ cả hai phía** (LC 1580).

In [30]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_prefix_min(warehouse):
    left = [warehouse[0]] * len(warehouse)
    for i in range(1, len(warehouse)):
        # TODO: left[i] = min của left[i-1] và warehouse[i]
        pass
    return left

In [31]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_assignment(boxes, warehouse):
    # TODO
    raise NotImplementedError

def ex5_both_sides(boxes, warehouse):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mỗi phòng một hộp. (2) Kho 2 phòng cao 1, hộp cao 1 ⇒ 2; hộp cao 5 không qua được phòng cao 4. (3) `2` và `0`.

**Bài 2.** (1) `eff = [6,4,4,2,2]`. (2) Hộp tăng dần `[1,2,3,5]`: 1 → phòng 4, 2 → phòng 3, 3 → phòng 2, 5 → phòng 1? `eff[1] = 4 < 5` ⇒ lùi tới phòng 0 (`6`) ✔. (3) Đáp án `4`.

**Bài 3.** (1) Duyệt từ trái. (2) `left[i] = min(left[i − 1], warehouse[i])`. (3) Hoặc `list(accumulate(warehouse, min))`.

**Bài 4.** (1) Chạy cách 2, mỗi lần đặt thì ghi `(bs[i], j)`. (2) Kiểm tra bằng `eff`. (3) Thêm kiểm tra bằng mô phỏng đẩy hộp theo thứ tự phòng sâu trước.

**Bài 5.** (1) Hộp tới phòng `i` từ trái cần `≤ min(wh[0..i])`, từ phải cần `≤ min(wh[i..n−1])`. (2) `eff[i] = max(trái, phải)`. (3) Ghép `sorted(eff)` với `sorted(boxes)` như cách 4. Kiểm chứng bằng `brute_force(..., both_sides=True)`.

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1
a1 = (Solution().maxBoxesInWarehouse([1, 1, 1], [1, 1]), Solution().maxBoxesInWarehouse([5], [4, 10]))
print("Bài 1:", a1)
assert a1 == (2, 0)

# Bài 2
a2 = trace_small_to_deep([2, 5, 3, 1], [6, 4, 4, 2, 3])
assert a2 == 4 == brute_force([2, 5, 3, 1], [6, 4, 4, 2, 3])

Bài 1: (2, 0)
left = [6, 4, 4, 2, 2], hộp tăng dần = [1, 2, 3, 5]
  hộp 1: bỏ qua phòng [], đặt vào phòng 4 (left=2)
  hộp 2: bỏ qua phòng [], đặt vào phòng 3 (left=2)
  hộp 3: bỏ qua phòng [], đặt vào phòng 2 (left=4)
  hộp 5: bỏ qua phòng [1], đặt vào phòng 0 (left=6)
  đáp án 4


In [33]:
# Bài 3
def ex3_prefix_min_sol(warehouse):
    left = [warehouse[0]] * len(warehouse)
    for i in range(1, len(warehouse)):
        left[i] = min(left[i - 1], warehouse[i])
    return left

rng3 = random.Random(3)
for _ in range(300):
    wh = [rng3.randint(1, 100) for _ in range(rng3.randint(1, 30))]
    assert ex3_prefix_min_sol(wh) == list(accumulate(wh, min))
print("Bài 3: OK", ex3_prefix_min_sol([5, 3, 3, 4, 1]))

Bài 3: OK [5, 3, 3, 3, 1]


In [34]:
# Bài 4
def ex4_assignment_sol(boxes, warehouse):
    n = len(warehouse)
    left = ex3_prefix_min_sol(warehouse)
    bs = sorted(boxes)
    pairs = []
    i, j = 0, n - 1
    while i < len(bs):
        while j >= 0 and left[j] < bs[i]:
            j -= 1
        if j < 0:
            break
        pairs.append((bs[i], j))
        i, j = i + 1, j - 1
    return pairs

def simulate_push(pairs, warehouse):
    # đẩy theo thứ tự phòng sâu trước; kiểm tra đường đi trống và đủ cao
    occ = [False] * len(warehouse)
    for h, p in sorted(pairs, key=lambda t: -t[1]):
        if any(occ[q] or warehouse[q] < h for q in range(p + 1)):
            return False
        occ[p] = True
    return True

rng4 = random.Random(4)
for _ in range(600):
    boxes = [rng4.randint(1, 8) for _ in range(rng4.randint(1, 8))]
    wh = [rng4.randint(1, 8) for _ in range(rng4.randint(1, 8))]
    pairs = ex4_assignment_sol(boxes, wh)
    assert len({p for _, p in pairs}) == len(pairs)
    assert simulate_push(pairs, wh)
    assert len(pairs) == Solution().maxBoxesInWarehouse(list(boxes), list(wh))
print("Bài 4: OK, ví dụ 1:", ex4_assignment_sol([4, 3, 4, 1], [5, 3, 3, 4, 1]))

Bài 4: OK, ví dụ 1: [(1, 4), (3, 3), (4, 0)]


In [35]:
# Bài 5 (LC 1580)
def ex5_both_sides_sol(boxes, warehouse):
    pre = list(accumulate(warehouse, min))
    suf = list(accumulate(warehouse[::-1], min))[::-1]
    caps = sorted(max(a, b) for a, b in zip(pre, suf))
    bs = sorted(boxes)
    b = 0
    for cap in caps:
        if b < len(bs) and bs[b] <= cap:
            b += 1
    return b

assert ex5_both_sides_sol([1, 2, 2, 3, 4], [3, 4, 1, 2]) == 4          # LC 1580 ví dụ 1
assert ex5_both_sides_sol([3, 5, 5, 2], [2, 1, 3, 4, 5]) == 3          # LC 1580 ví dụ 2
rng5 = random.Random(5)
for _ in range(800):
    boxes = [rng5.randint(1, 7) for _ in range(rng5.randint(1, 5))]
    wh = [rng5.randint(1, 7) for _ in range(rng5.randint(1, 5))]
    assert ex5_both_sides_sol(boxes, wh) == brute_force(boxes, wh, both_sides=True), (boxes, wh)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def maxBoxesInWarehouse(self, boxes, warehouse):
        n = len(warehouse)
        left = [warehouse[0]] * n
        for i in range(1, n):
            left[i] = min(left[i - 1], warehouse[i])     # sức chứa hiệu dụng
        boxes.sort()
        i, j = 0, n - 1
        while i < len(boxes):
            while j >= 0 and left[j] < boxes[i]:
                j -= 1
            if j < 0:
                break
            i, j = i + 1, j - 1                          # hộp nhỏ nhất -> phòng sâu nhất còn vừa
        return i
```

**Template "đi vào từ một đầu + ghép cặp":** min tiền tố ⇒ sức chứa không tăng ⇒ sắp hộp ⇒ hai con trỏ ngược chiều (hoặc Assign Cookies trên `eff[::-1]`).

**Checklist trước khi submit:**
- [ ] So với **min tiền tố**, không phải `warehouse[j]`.
- [ ] Hộp nhỏ nhất ↔ phòng **sâu nhất** (hoặc hộp lớn nhất ↔ phòng ngoài cùng, không vừa thì bỏ hộp).
- [ ] Dừng khi `j < 0`.
- [ ] Thử nhanh: ba ví dụ đề → `3, 3, 1`; `([4],[3,5]) → 0`; `([1,1,1],[1,1]) → 2`.